# Manifold Learning on a Swiss Roll — PCA vs LLE

The **Swiss roll** is the classic teaching example for *manifold learning*. It is a 2D sheet of paper that has been rolled up and embedded in 3D space. Even though every point lives in $\mathbb{R}^3$, the data really only has **two intrinsic degrees of freedom**: *how far along the roll* and *how far across the width*. Manifold learning tries to recover those two coordinates — i.e. to **unroll** the sheet back into a flat 2D strip.

We compare two dimensionality-reduction methods on this data:

1. **PCA** (Principal Component Analysis) — a *linear* method. It can only rotate and project; it cannot bend space. We apply it to the **first 1000** samples.
2. **LLE** (Locally Linear Embedding) — a *non-linear*, neighbour-preserving method. It can follow the curved sheet and flatten it. We apply it to the **second 1000** samples.

The punchline: PCA *flattens* the roll (distant parts of the sheet end up on top of each other), while LLE *unrolls* it (the colour gradient becomes a smooth left-to-right strip).

In [ ]:
from sklearn.datasets import make_swiss_roll   # generates the rolled-up 2D sheet in 3D
import matplotlib.pyplot as plt                # all plotting (incl. the 3D scatters)
import numpy as np                             # array math (used for the PCA error sum)

# NOTE: there is deliberately NO warnings.filterwarnings(...) here. Every plotting cell
# below builds its own figure/axes, and both PCA and LLE run cleanly on this data, so the
# notebook executes top-to-bottom with no warnings to hide.

## 1. What is a manifold?

A **manifold** is a shape that *locally* looks flat (Euclidean) even if *globally* it is curved. The surface of the Earth is a 2D manifold living in 3D: any small patch looks like a flat map, but the whole thing is a sphere.

The Swiss roll is the same idea: it is an intrinsically **2-dimensional** surface that has been curled into **3-dimensional** space. The *ambient* dimension is 3, but the *intrinsic* dimension is 2.

The **manifold hypothesis** says that high-dimensional real-world data (images, sensor readings, ...) usually lies on or near a much lower-dimensional manifold. Manifold learning aims to find coordinates *on* that manifold — a faithful low-dimensional description — rather than just projecting the raw ambient coordinates.

`make_swiss_roll` returns:
- `x`: the 3D coordinates, shape `(n_samples, 3)`.
- `color`: a 1D value that increases *along the length of the roll*. We use it purely to colour points so we can see whether a method keeps the sheet's ordering intact.

In [ ]:
# n_samples points sampled on the rolled sheet; random_state fixes the sample so the
# notebook is reproducible. color[i] grows as you travel ALONG the roll (its true 1st axis).
x, color = make_swiss_roll(n_samples=2000, random_state=42)

print(f"x shape: {x.shape}   (ambient dimension = 3)")
print(f"color shape: {color.shape}   (position along the length of the roll)")

### Look at the raw roll in 3D

Colour encodes position along the length of the sheet. Notice how the colour spirals around: points that are *far apart along the sheet* can be *physically close in 3D* (adjacent layers of the roll). That spiral is exactly what makes the problem hard for a linear method.

In [ ]:
fig = plt.figure(figsize=(12, 8))            # own figure so this cell is self-contained
ax = fig.add_subplot(projection='3d')        # 3D axes: x has THREE columns to plot

# 3D scatter: pass all three coordinate columns as x, y, z. c=color tints each point by its
# position along the roll; the viridis colormap turns those numbers into colours.
ax.scatter(x[:, 0], x[:, 1], x[:, 2], c=color, cmap=plt.cm.viridis)
ax.set_title('Raw Swiss roll (all 2000 samples, 3D)')
plt.show()

## 2. PCA — a linear projection (why it *cannot* unroll)

PCA finds the directions of maximum variance and projects the data onto the top ones. Every PCA output coordinate is a **linear combination** of the input coordinates:

$$ x_{\text{2D}} = W^{\top}(x - \bar{x}), \qquad W \in \mathbb{R}^{3 \times 2} $$

Because the map $W^{\top}$ is linear, PCA can only **rotate and flatten** the cloud — it slices the roll with a flat plane and drops one dimension. It has no way to *bend* space, so it cannot separate the layers of the roll.

The consequence: two points on opposite ends of the sheet, which happen to sit near the same projection plane, get **mapped on top of each other**. The colour gradient folds over itself instead of laying out smoothly.

We run PCA on the **first 1000 samples**.

In [ ]:
# Use the FIRST 1000 samples for the PCA demo (LLE later uses the second 1000).
x_pca_data = x[:1000]        # (1000, 3) input coordinates
color_pca = color[:1000]     # (1000,)  matching colours (position along the roll)

In [ ]:
fig = plt.figure(figsize=(12, 8))            # own figure/axes for this cell
ax = fig.add_subplot(projection='3d')        # 3D view of the PCA INPUT before reduction

# Same 3D scatter as before, now on just the first-1000 subset PCA will see.
ax.scatter(x_pca_data[:, 0], x_pca_data[:, 1], x_pca_data[:, 2], c=color_pca, cmap=plt.cm.viridis)
ax.set_title('PCA input — first 1000 samples (3D, before reduction)')
plt.show()

### Fitting PCA and its error

PCA keeps the top-2 variance directions. The `explained_variance_ratio_` tells us the fraction of total variance each kept component captures, so a natural **reconstruction error** is the fraction of variance we *threw away* by dropping the 3rd dimension:

$$ \text{PCA error} = 1 - \sum_{k=1}^{2} \frac{\lambda_k}{\sum_j \lambda_j} $$

This is a **global variance** measure — how much of the cloud's spread the flat projection failed to preserve.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)                 # keep the 2 highest-variance directions
x_pca = pca.fit_transform(x_pca_data)     # (1000, 2) linear projection onto those directions

# Reconstruction error = variance NOT captured by the 2 kept components (the dropped 3rd axis).
pca_error = 1 - np.sum(pca.explained_variance_ratio_)
print(f"PCA reconstruction error (variance lost) = {pca_error:.4f}")

In [ ]:
fig = plt.figure(figsize=(12, 8))         # own 2D figure for the projection result
ax = fig.add_subplot()                     # plain 2D axes: x_pca has only 2 columns

# 2D scatter of the projected points, coloured by their true position along the roll.
# Watch the colours FOLD OVER each other -> PCA flattened the roll instead of unrolling it.
ax.scatter(x_pca[:, 0], x_pca[:, 1], c=color_pca, cmap=plt.cm.viridis)
ax.set_title(f'PCA projection - first 1000 samples (reconstruction error = {pca_error:.4f})')
plt.show()

## 3. LLE — non-linear, neighbour-preserving (it *can* unroll)

**Locally Linear Embedding** starts from the manifold assumption that each point is (approximately) a **linear combination of its nearest neighbours**. Concretely:

1. For each point $x_i$, find its $k$ nearest neighbours and solve for weights $W_{ij}$ that best reconstruct $x_i$ from those neighbours:

$$ \min_{W} \sum_i \Big\| x_i - \sum_{j \in \mathcal{N}(i)} W_{ij}\, x_j \Big\|^2, \quad \text{s.t. } \sum_j W_{ij} = 1 $$

2. Then find **low-dimensional** coordinates $y_i$ that are reconstructed by the *same* weights:

$$ \min_{Y} \sum_i \Big\| y_i - \sum_{j \in \mathcal{N}(i)} W_{ij}\, y_j \Big\|^2 $$

Because the weights are computed only from **local neighbourhoods**, LLE preserves the local geometry while being free to reshape the global layout. On the Swiss roll that means it can peel the layers apart and lay the sheet flat — something the linear PCA map can never do.

We run LLE on the **second 1000 samples**, with `n_neighbors=12` (the size of each local patch).

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

# Second 1000 samples -> LLE (kept separate from the PCA subset).
x_lle_data = x[1000:]        # (1000, 3) input coordinates for LLE
color_lle = color[1000:]     # (1000,)  matching colours (position along the roll)

In [ ]:
fig = plt.figure(figsize=(12, 8))            # own figure/axes for this cell
ax = fig.add_subplot(projection='3d')        # 3D axes: x_lle_data has THREE columns

# BUG FIX: the raw LLE input is 3D, so it MUST use a 3D scatter (ax.scatter with x, y, z).
# The old code called the 2D plt.scatter(...) with three positional arrays, which made
# matplotlib read the z-column as the marker-size 's' argument ->
# 'ValueError: s must be a float array-like...'. Plotting on 3D axes fixes it.
ax.scatter(x_lle_data[:, 0], x_lle_data[:, 1], x_lle_data[:, 2], c=color_lle, cmap=plt.cm.viridis)
ax.set_title('LLE input — second 1000 samples (3D, before reduction)')
plt.show()

### Fitting LLE and its error

After fitting, `reconstruction_error_` reports the residual of step 2 — how well the low-dimensional embedding is reconstructed from each point's neighbours using the locally-fitted weights. It is a **local neighbour-reconstruction cost**, not a variance measure.

In [ ]:
# n_components=2 -> unroll to a flat sheet. n_neighbors=12 -> each point is reconstructed
# from its 12 nearest neighbours (the 'local patch' size; too small = noisy, too big = it
# starts to act globally/linearly).
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=12)
x_lle = lle.fit_transform(x_lle_data)     # (1000, 2) unrolled coordinates

# Local neighbour-reconstruction cost of the embedding (NOT comparable to PCA's error).
lle_error = lle.reconstruction_error_
print(f"LLE reconstruction error (local neighbour cost) = {lle_error:.3e}")

In [ ]:
fig = plt.figure(figsize=(12, 8))         # own 2D figure for the embedding result
ax = fig.add_subplot()                     # plain 2D axes: x_lle has only 2 columns

# 2D scatter of the unrolled coordinates. Now the colour varies SMOOTHLY across the strip
# (no fold-over): LLE recovered the sheet's intrinsic left-to-right ordering.
ax.scatter(x_lle[:, 0], x_lle[:, 1], c=color_lle, cmap=plt.cm.viridis)
ax.set_title(f'LLE projection - second 1000 samples (reconstruction error = {lle_error:.3e})')
plt.show()

## 4. Why the two error numbers are *not* comparable

It is tempting to line up `pca_error` and `lle_error` and declare a winner, but they measure completely different things:

| | PCA error | LLE error |
|---|---|---|
| **What it measures** | Fraction of total **variance lost** when dropping to 2D | **Local neighbour-reconstruction** residual of the embedding |
| **Scope** | Global spread of the cloud | Local patches ($k$ neighbours) |
| **Formula** | $1 - \sum_{k\le 2}\lambda_k / \sum_j \lambda_j$ | $\sum_i \lVert y_i - \sum_j W_{ij} y_j \rVert^2$ |

They live on different scales and answer different questions, so a smaller number for one does **not** mean it is the better method. Judge each by the *right* criterion — and, for a manifold like this, by whether the sheet actually got unrolled (look at the plots, not just the numbers).

## Summary

- Generated a **Swiss roll** (2000 points, 3D) — a classic non-linear manifold.
- Applied **PCA** (linear) on the first 1000 samples and **LLE** (non-linear) on the second 1000.
- **PCA** just flattens the roll → distant parts of the sheet overlap; the color gradient folds over itself.
- **LLE** unrolls the manifold → color varies smoothly across a flat 2D strip; local neighborhoods are preserved.

**Key takeaway:** the two error numbers are *not comparable* — PCA's is variance lost in projection, LLE's is the local neighbor-reconstruction cost. For manifold data like this, **LLE preserves the intrinsic structure that linear PCA cannot**.